In [ ]:
import polars as pl

In [ ]:
markets = pl.read_parquet('data/processed/markets_political.parquet')

In [ ]:
m_list = markets['condition_id'].to_list()

In [ ]:
df = (
    pl.scan_parquet('data/raw/quant.parquet', parallel='prefiltered')
    .filter(pl.col('condition_id').str.contains_any(m_list))
    .collect()
)

In [ ]:
df.write_parquet('data/processed/quant-filtered.parquet')

In [ ]:
print(df.shape)
print(df.columns)
print(df.dtypes)

# Date range
print(df['timestamp'].min())
print(df['timestamp'].max())

# Unique markets captured
print(f"Unique markets: {df['condition_id'].n_unique()}")
print(f"Expected: {len(m_list)}")

# Trades per market distribution
print(df.group_by('condition_id').len()['len'].describe())

In [ ]:
import datetime
print(datetime.datetime.fromtimestamp(1669060169))  # earliest
print(datetime.datetime.fromtimestamp(1773657371))  # latest

In [ ]:
missing = set(m_list) - set(df['condition_id'].to_list())
missing_markets = markets.filter(pl.col('condition_id').is_in(list(missing)))
print(missing_markets['resolved_yes'].value_counts())
print(missing_markets.select(['question']).sample(5))

In [ ]:
df = df.with_columns(
    pl.from_epoch(pl.col('timestamp'), time_unit='s').alias('datetime')
).with_columns(
    pl.col('datetime').dt.convert_time_zone('UTC').dt.cast_time_unit('ms')
)
print(df['datetime'].min())
print(df['datetime'].max())

## Feature Matrix Analysis

In [ ]:
import polars as pl

In [ ]:
df = pl.read_parquet('data/processed/feature_matrix.parquet')

In [ ]:
df

In [ ]:
feature_matrix = pl.read_parquet('data/processed/feature_matrix.parquet')

print(feature_matrix.shape)
print(feature_matrix.null_count())
print(feature_matrix.describe())
print(feature_matrix['resolved_yes'].value_counts())
print(feature_matrix.head(5))

In [ ]:
markets = pl.read_parquet('data/processed/markets_political.parquet')

feature_matrix = feature_matrix.join(
    markets.select(['condition_id', 'end_date']),
    left_on='market_id',
    right_on='condition_id',
    how='left'
)

In [ ]:
feature_matrix.write_parquet('data/processed/feature_matrix.parquet')

In [ ]:
# What is the correlation between price_end and resolved_yes?
import numpy as np
test_df = pl.read_parquet('data/model/test.parquet')
correlation = np.corrcoef(
    test_df['price_end'].to_numpy(),
    test_df['resolved_yes'].cast(pl.Int32).to_numpy()
)[0,1]
print(f"price_end vs resolved_yes correlation: {correlation:.4f}")